# 📓 Semana 3 · Dia 2 — Leitura e escrita: formatos, opções e schema

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA (ELT) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Leitura correta dos 4 formatos com schema explícito |

---


## 📖 Teoria — Schema enforcement vs inferência

Quando você lê um CSV sem schema, o Spark **infere** os tipos — mas isso pode errar (ex.: `000123` vira INT perdendo zeros à esquerda).

Em produção, defina o **schema explícito**: mais rápido, determinístico e seguro. No Delta, o schema fica gravado na tabela — sempre enforcement.


## 📖 Teoria — Paths: DBFS vs Volumes

**DBFS**: `dbfs:/FileStore/...` — legado, mas prático para arquivos de estudo.
**Volumes**: `/Volumes/workspace/bronze/vol_dados_curso/...` — o padrão 2026, governado pelo UC.

> 🎯 **Dica de prova**: a DEA 2026 cobre a recomendação de usar **Volumes** para arquivos e a diferença entre paths de Volume e DBFS.


### 💻 Na prática — Schema explícito

Defina o schema do CSV do projeto com `StructType` e leia com ele.


In [ ]:
# Schema explícito do CSV de vendas
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType,
                               DoubleType, TimestampType)
schema_vendas = StructType([
    StructField("InvoiceNo", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("InvoiceDate", StringType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("CustomerID", StringType(), True),
    StructField("Country", StringType(), True),
])
df = (spark.read
    .format("csv")
    .option("header", True)
    .option("multiLine", True)
    .schema(schema_vendas)
    .load("/Volumes/workspace/bronze/vol_dados_curso/vendas.csv"))
df.printSchema()
print("Linhas:", df.count())

### 💻 Na prática — Escrita com modo

O parâmetro `mode` decide o que fazer quando o destino existe: `overwrite`, `append`, `error`, `ignore`.


In [ ]:
# Escrita nos modos comuns (no Volume do curso)
df.limit(1000).write.mode("overwrite").format("parquet").save("/Volumes/workspace/bronze/vol_dados_curso/vendas_amostra")
df.limit(100).write.mode("append").format("parquet").save("/Volumes/workspace/bronze/vol_dados_curso/vendas_amostra")
print("Amostra gravada e estendida (overwrite + append).")

In [ ]:
# Volumes: o caminho moderno
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.bronze.vol_dados_curso")
path_volume = "/Volumes/workspace/bronze/vol_dados_curso/vendas.parquet"
df.limit(1000).write.mode("overwrite").format("parquet").save(path_volume)
print("Gravado no Volume:", path_volume)

> 🎯 **Dica de prova**: `mode('overwrite')` em tabela managed pode apagar dados se o schema mudar — na prova, prefira `overwrite` explícito com schema compatível, e lembre: em pipelines de Bronze use **append** (append-only).


## 🎯 Exercícios de fixação

**1.** Leia o JSON do projeto (`/Volumes/workspace/bronze/vol_dados_curso/vendas.json`) com schema explícito.

**2.** Qual opção usa `header` e `multiLine` para CSV?

**3.** Escreva 1000 linhas em Parquet num Volume e leia de volta contando.


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** JSON com schema

`spark.read.schema(schema_vendas).json('/Volumes/workspace/bronze/vol_dados_curso/vendas.json')` — para JSON o caminho de um arquivo único, sem header.

**2.** Opções CSV

`.option('header', True)` (primeira linha é cabeçalho) e `.option('multiLine', True)` (quebra de linha dentro de campos entre aspas).

**3.** Volume

`df.write.mode('overwrite').format('parquet').save('/Volumes/workspace/bronze/vol_dados_curso/vendas.parquet')` e depois `spark.read.parquet(...)` contando linhas.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*